# Post Hoc Analysis Script

This is code used to produce embeddings for each of the models we trained. The code is fairly straight forward as we do two things:

1. Load the dataset.
2. Load and run analysis on models.

- For each model, we do the following steps:

1. Create a class instance of the model.
2. Load the state dict of the model.
3. Create the umap embededings of embeddings in the penaltimate layers.
4. Create saliency maps for each model.
5. Save each of these in the results section.

In [ ]:
import os
import torch
import numpy as np
import plotly.express as px

from umap import UMAP

from captum.attr import Saliency

from torch_brain.utils import seed_everything

seed_everything(0)

In [ ]:
base_dir = os.getcwd().split("additional_scripts")[0]
device = "callable" if torch.cuda.is_available() else "cpu"

print(f"Using: {device}")

## Load Data

In [ ]:
from neural_decode.dataset.transformer_dataloader import get_train_val_loaders

from neural_decode.models.hopfield_only import HopfieldOnlyDecoder
from neural_decode.models.transformer import TransformerNeuralDecoder 
from neural_decode.models.transformer_hopfield import TransformerHopfieldDecoder

from neural_decode.post_hoc_analysis.embedding_analysis import compute_umap_embeddings
from neural_decode.post_hoc_analysis.saliency_based_analysis import compute_and_return_attribution_maps

In [ ]:
path_to_neural_dataset = os.path.join(base_dir, "data", "perich_miller_population_2018", "t_20130819_center_out_reaching")
train_dataset, train_loader, val_dataset, val_loader = get_train_val_loaders(path_to_neural_dataset, batch_size=64)

num_units = len(train_dataset.get_unit_ids())
print(f"Num Units in Session: {num_units}")

In [ ]:
path_to_models = os.path.join(base_dir, "results", "models")

tf_save_file = os.path.join(path_to_models, "transformer_model.pth")
hopfield_only_save_file = os.path.join(path_to_models, "hopfield_only_model.pth")
tf_hopfield_save_file = os.path.join(path_to_models, "transformer_hopfield_model.pth")

## Transformer Embedding and Saliency Map Collection

In [ ]:
tf_model = TransformerNeuralDecoder(
    num_units=num_units,
    bin_size=10e-3,
    sequence_length=1.0,
    dim_output=2,
    dim_hidden=128,
    n_layers=3,
    n_heads=4,
).to(device)

# Load state dicts for both models.
tf_model.load_state_dict(torch.load(tf_save_file, map_location=device))

# Add tokenizer for train and validation loader.
train_dataset.transform = tf_model.tokenize
val_dataset.transform = tf_model.tokenize

## Collecting umap and attribution maps.
tf_embeddings, tf_labels = compute_umap_embeddings(tf_model, val_loader)
normed_saliency_maps_tf = compute_and_return_attribution_maps(tf_model, val_loader)


In [ ]:
sv_file_for_tf_emb = os.path.join(base_dir, "results", "embeddings", "tf", "transformer_umap_embeddings.npz")
sv_file_for_tf_lbs = os.path.join(base_dir, "results", "embeddings", "tf", "transformer_umap_labels.npz")

np.save(sv_file_for_tf_emb, tf_embeddings)
np.save(sv_file_for_tf_lbs, tf_labels)

In [ ]:
sv_file_for_tf_sal = os.path.join(base_dir, "results", "saliency_maps", "tf", "transformer_saliency_maps.npz")
np.save(sv_file_for_tf_sal, normed_saliency_maps_tf)

## Hopfield Only Embedding and Saliency Map Collection

In [ ]:
hopfield_only_model = HopfieldOnlyDecoder(
    num_units=num_units,
    bin_size=10e-3,
    sequence_length=1.0,
    dim_output=2,
    dim_hidden=128,
    n_layers=3,
).to(device)

# Load state dicts for both models.
hopfield_only_model.load_state_dict(torch.load(hopfield_only_save_file, map_location=device))

# Add tokenizer for train and validation loader.
train_dataset.transform = hopfield_only_model.tokenize
val_dataset.transform = hopfield_only_model.tokenize

## Collecting umap and attribution maps.
hopfield_embeddings, hopfield_labels = compute_umap_embeddings(hopfield_only_model, val_loader)
normed_saliency_maps_hopfield = compute_and_return_attribution_maps(hopfield_only_model, val_loader)


In [ ]:
sv_file_for_hf_emb = os.path.join(base_dir, "results", "embeddings", "hopfield_only", "hopfield_umap_embeddings.npz")
sv_file_for_hf_lbs = os.path.join(base_dir, "results", "embeddings", "hopfield_only", "hopfield_umap_labels.npz")

np.save(sv_file_for_hf_emb, hopfield_embeddings)
np.save(sv_file_for_hf_lbs, hopfield_labels)

In [ ]:
sv_file_for_tf_sal = os.path.join(base_dir, "results", "saliency_maps", "hopfield_only", "transformer_saliency_maps.npz")
np.save(sv_file_for_tf_sal, normed_saliency_maps_hopfield)

# TF + Hopfiled Embeddings and Saliency Map Collection

In [ ]:
tf_hopfield_model = TransformerHopfieldDecoder(
    num_units=num_units,
    bin_size=10e-3,
    sequence_length=1.0,
    dim_output=2,
    dim_hidden=128,
    n_layers=3,
    n_heads=4,
).to(device)

# Load state dicts for both models.
tf_hopfield_model.load_state_dict(torch.load(tf_hopfield_save_file, map_location=device))

# Add tokenizer for train and validation loader.
train_dataset.transform = tf_hopfield_model.tokenize
val_dataset.transform = tf_hopfield_model.tokenize

## Collecting umap and attribution maps.
tf_hopfield_embeddings, tf_hopfield_labels = compute_umap_embeddings(tf_hopfield_model, val_loader)
normed_saliency_maps_tf_hopfield = compute_and_return_attribution_maps(tf_hopfield_model, val_loader)

In [ ]:
sv_file_for_hf_emb = os.path.join(base_dir, "results", "embeddings", "tf_hopfield", "hopfield_umap_embeddings.npz")
sv_file_for_hf_lbs = os.path.join(base_dir, "results", "embeddings", "tf_hopfield", "hopfield_umap_labels.npz")

np.save(sv_file_for_hf_emb, tf_hopfield_embeddings)
np.save(sv_file_for_hf_lbs, tf_hopfield_labels)

In [ ]:
sv_file_for_tf_sal = os.path.join(base_dir, "results", "saliency_maps", "tf_hopfield", "transformer_saliency_maps.npz")
np.save(sv_file_for_tf_sal, normed_saliency_maps_tf_hopfield)